# 1. Imports and Setup

In [1]:
import glob
import os
import pandas as pd
import numpy as np
from collections import Counter
from joblib import load, dump
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.inspection import permutation_importance
from scipy.stats import entropy

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

# 2. Data Loading & Feature Engineering

In [2]:
labels_folder = "data/Merged_labels"
use_feature_engineering = True

rivers_df = pd.read_csv("data/rivers.csv", sep=';')
features_base = ["DIST_DN_KM", "DIS_AV_CMS", "ORD_CLAS", "ORD_FLOW", "ORD_STRA", "DIST_UP_KM", "UPLAND_SKM", "CATCH_SKM"]
rivers_features = rivers_df[["HYRIV_ID"] + features_base].copy()

def engineer_features(df):
    df = df.copy()
    df['NET_LENGTH'] = df['DIST_UP_KM'] + df['DIST_DN_KM']
    df['ORDER_RATIO'] = df['ORD_FLOW'] / (df['ORD_CLAS'] + 1e-5)
    df['LENGTH_OVER_ORDER'] = df['NET_LENGTH'] / (df['ORD_FLOW'] + 1e-5)
    df['UPLAND_DENSITY'] = df['UPLAND_SKM'] / (df['DIS_AV_CMS'] + 1e-5)
    return df

if use_feature_engineering:
    rivers_features = engineer_features(rivers_features)
    final_features = ["DIST_DN_KM", "NET_LENGTH", "ORDER_RATIO", "LENGTH_OVER_ORDER", "UPLAND_DENSITY"]

features = final_features

labelled_dfs = []
label_map = {}


for file in glob.glob(f"{labels_folder}/*.csv"):
    if file.lower() == "data/rivers.csv" or file.lower().endswith("new.csv"):
        continue
    class_name = os.path.splitext(os.path.basename(file))[0].capitalize()
    df = pd.read_csv(file, sep=';')
    
    df = df[["HYRIV_ID"]]
    df["label"] = class_name
    labelled_dfs.append(df)
    label_map[class_name] = file

labels_df = pd.concat(labelled_dfs, ignore_index=True)
labelled_df = pd.merge(rivers_features, labels_df, on="HYRIV_ID", how="inner")

X_labelled_raw = labelled_df[final_features].values
y_labelled = labelled_df["label"].values

imputer = SimpleImputer(strategy="mean")
scaler = StandardScaler()


print(f"Total labeled rivers found: {len(X_labelled_raw)}")


Total labeled rivers found: 9907


# 3. Train / Test Split and Class Weights

In [3]:
le = LabelEncoder()
y_labelled_encoded = le.fit_transform(y_labelled)

def decode_labels(encoded_labels):
    return le.inverse_transform(encoded_labels)

id_labelled = labelled_df["HYRIV_ID"].values

X_train_raw, X_test_raw, y_train, y_test, id_train, id_test = train_test_split(
    X_labelled_raw, y_labelled_encoded, id_labelled,
    test_size=0.2, random_state=42, stratify=y_labelled_encoded
)

X_train_imputed = imputer.fit_transform(X_train_raw)
X_train = scaler.fit_transform(X_train_imputed)
X_test_imputed = imputer.transform(X_test_raw)
X_test = scaler.transform(X_test_imputed)

test_hyriv_ids = id_test.tolist()

y_encoded_all = le.transform(y_labelled)
classes_encoded = np.unique(y_encoded_all)
class_weights = compute_class_weight(class_weight='balanced', classes=classes_encoded, y=y_encoded_all)
class_weight_dict = dict(zip(classes_encoded, class_weights))

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 7925, Test size: 1982


# 4. Model Definition & Pseudo-Labeling Setup

In [4]:
def train_models(
    model_type="rf",
    num_iterations=3,
    model_save_path="final_model.joblib",
    threshold=0.95,
    class_thresholds=None,
    min_new_labels=500,
    max_per_class=None,
    test_hyriv_ids=None,
):
    labelled_local = labelled_df.copy()
    X_labelled_imp = imputer.transform(labelled_local[features])
    X_labelled_scaled = scaler.transform(X_labelled_imp)
    y_encoded = le.transform(labelled_local["label"])
    id_labelled = labelled_local["HYRIV_ID"].values
    
    mask_train = ~np.isin(id_labelled, test_hyriv_ids)
    labelled_ids = set(id_labelled[mask_train])

    curr_X_train = X_labelled_scaled[mask_train]
    curr_y_train = y_encoded[mask_train]
    
    if model_type == "rf":
        model = RandomForestClassifier(
            n_estimators=500, max_depth=20, min_samples_leaf=2, max_features=0.5,
            class_weight=class_weight_dict, random_state=42, n_jobs=-1, bootstrap=False
        )
    elif model_type == "xgb":
        model = XGBClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=8, subsample=0.8,
            colsample_bytree=0.8, objective="multi:softprob",
            num_class=len(np.unique(y_encoded)), eval_metric='mlogloss',
            random_state=42, verbosity=0
        )
    elif model_type == "lgbm":
        model = LGBMClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=8, num_leaves=31,
            subsample=0.8, colsample_bytree=0.8, class_weight="balanced",
            random_state=42, verbose=-1
        )
    elif model_type == "hist_gb":
        model = HistGradientBoostingClassifier(
            learning_rate=0.05, max_iter=500, max_depth=8, random_state=42
        )

    label_counts = [len(curr_y_train)]

    for iteration in range(num_iterations):
        model.fit(curr_X_train, curr_y_train)

        all_ids = set(rivers_features["HYRIV_ID"])
        test_ids = set(test_hyriv_ids) if test_hyriv_ids else set()
        excluded_ids = labelled_ids.union(test_ids)
        unlabeled_ids = list(all_ids - excluded_ids)

        if not unlabeled_ids: break

        unlabeled_df = rivers_features[rivers_features["HYRIV_ID"].isin(unlabeled_ids)].copy()
        X_unlabeled = scaler.transform(imputer.transform(unlabeled_df[features].values))
        y_proba = model.predict_proba(X_unlabeled)
        y_pred = model.predict(X_unlabeled)

        confident_indices = []
        for i, (proba_row, pred_idx) in enumerate(zip(y_proba, y_pred)):
            class_name = le.inverse_transform([pred_idx])[0]
            class_key = class_name.split("\\")[-1].lower()
            class_thresh = class_thresholds.get(class_key, threshold) if class_thresholds else threshold
            if proba_row[pred_idx] >= class_thresh:
                confident_indices.append(i)

        if len(confident_indices) < min_new_labels: break
        print(f"Iteration {iteration+1}: added {len(confident_indices)} pseudo-labels. Train size: {len(curr_y_train) + len(confident_indices)}")

        confident_indices = np.array(confident_indices)
        X_confident = X_unlabeled[confident_indices]
        y_confident = y_pred[confident_indices]
        ids_confident = unlabeled_df.iloc[confident_indices]["HYRIV_ID"].values

        if max_per_class:
            filtered_X, filtered_y, filtered_ids = [], [], []
            counter = Counter()
            for x, y, id_ in zip(X_confident, y_confident, ids_confident):
                name = le.inverse_transform([y])[0]
                if counter[name] < max_per_class:
                    filtered_X.append(x)
                    filtered_y.append(y)
                    filtered_ids.append(id_)
                    counter[name] += 1
            X_confident = np.array(filtered_X)
            y_confident = np.array(filtered_y)
            ids_confident = np.array(filtered_ids)

        curr_X_train = np.vstack([curr_X_train, X_confident])
        curr_y_train = np.concatenate([curr_y_train, y_confident])
        labelled_ids.update(ids_confident)

    model.fit(curr_X_train, curr_y_train)
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    dump(model, model_save_path)
    
    return model

# 5. Base Models

In [5]:
base_dir = "improve_metrics/base_models"
rf_base = train_models("rf", num_iterations=0, model_save_path=f"{base_dir}/rf_base.joblib", test_hyriv_ids=test_hyriv_ids)
xgb_base = train_models("xgb", num_iterations=0, model_save_path=f"{base_dir}/xgb_base.joblib", test_hyriv_ids=test_hyriv_ids)
lgbm_base = train_models("lgbm", num_iterations=0, model_save_path=f"{base_dir}/lgbm_base.joblib", test_hyriv_ids=test_hyriv_ids)
hgb_base = train_models("hist_gb", num_iterations=0, model_save_path=f"{base_dir}/hist_gb_base.joblib", test_hyriv_ids=test_hyriv_ids)


# 6. Pseudo-Labeled Models

In [6]:
pseudo_dir = "improve_metrics/pseudo_models"
rf_thresholds = {'angular': 0.98, 'pinnate': 0.97, 'trellis': 0.88, 'parallel': 0.89, 'rectangular': 0.87, 'dendritic': 0.82, 'sub-dendritic': 0.80}
xgb_thresholds = {'angular': 0.95, 'pinnate': 0.95, 'trellis': 0.95, 'parallel': 0.90, 'rectangular': 0.90, 'dendritic': 0.85, 'sub-dendritic': 0.85}
lgbm_thresholds = {'angular': 0.96, 'pinnate': 0.96, 'trellis': 0.93, 'parallel': 0.91, 'rectangular': 0.89, 'dendritic': 0.85, 'sub-dendritic': 0.83}
hgb_thresholds = {'angular': 0.98, 'pinnate': 0.98, 'trellis': 0.98, 'parallel': 0.95, 'rectangular': 0.90, 'dendritic': 0.90, 'sub-dendritic': 0.90}

rf_pseudo = train_models("rf", num_iterations=3, model_save_path=f"{pseudo_dir}/rf_pseudo.joblib", class_thresholds=rf_thresholds, min_new_labels=1500, max_per_class=2000, test_hyriv_ids=test_hyriv_ids)
xgb_pseudo = train_models("xgb", num_iterations=3, model_save_path=f"{pseudo_dir}/xgb_pseudo.joblib", class_thresholds=xgb_thresholds, min_new_labels=500, max_per_class=1000, test_hyriv_ids=test_hyriv_ids)
lgbm_pseudo = train_models("lgbm", num_iterations=3, model_save_path=f"{pseudo_dir}/lgbm_pseudo.joblib", class_thresholds=lgbm_thresholds, min_new_labels=500, max_per_class=1000, test_hyriv_ids=test_hyriv_ids)
hgb_pseudo = train_models("hist_gb", num_iterations=3, model_save_path=f"{pseudo_dir}/hist_gb_pseudo.joblib", class_thresholds=hgb_thresholds, min_new_labels=500, max_per_class=1000, test_hyriv_ids=test_hyriv_ids)

pseudo_models = [rf_pseudo, xgb_pseudo, lgbm_pseudo, hgb_pseudo]


Iteration 1: added 6678 pseudo-labels. Train size: 14123
Iteration 2: added 3926 pseudo-labels. Train size: 17674
Iteration 3: added 2593 pseudo-labels. Train size: 20267
Iteration 1: added 21758 pseudo-labels. Train size: 29203
Iteration 2: added 20151 pseudo-labels. Train size: 34416
Iteration 3: added 16874 pseudo-labels. Train size: 37934
Iteration 1: added 24564 pseudo-labels. Train size: 32009
Iteration 2: added 22635 pseudo-labels. Train size: 37080
Iteration 3: added 18706 pseudo-labels. Train size: 40151
Iteration 1: added 22828 pseudo-labels. Train size: 30273
Iteration 2: added 16826 pseudo-labels. Train size: 31189
Iteration 3: added 12501 pseudo-labels. Train size: 33830


In [7]:
rf_p = rf_pseudo.predict_proba(X_test)
xgb_p = xgb_pseudo.predict_proba(X_test)
lgbm_p = lgbm_pseudo.predict_proba(X_test)
hgb_p = hgb_pseudo.predict_proba(X_test)

mean_p = (rf_p + xgb_p + lgbm_p + hgb_p) / 4
y_pred = np.argmax(mean_p, axis=1)

classes = le.inverse_transform(np.arange(len(np.unique(y_test))))
class_names = [c.split("\\")[-1].capitalize() for c in classes]

print("--- Ensemble Metrics on Test Set (Segments) ---")
print(classification_report(y_test, y_pred, target_names=class_names))


--- Ensemble Metrics on Test Set (Segments) ---
               precision    recall  f1-score   support

      Angular       0.95      0.89      0.92       277
    Dendritic       0.66      0.81      0.73       231
     Parallel       0.74      0.69      0.71       223
      Pinnate       0.92      0.91      0.92       518
  Rectangular       0.77      0.77      0.77       263
Sub-dendritic       0.67      0.65      0.66       168
      Trellis       0.85      0.83      0.84       302

     accuracy                           0.82      1982
    macro avg       0.79      0.79      0.79      1982
 weighted avg       0.82      0.82      0.82      1982



# 7. Ensemble & CMS Tools

In [8]:
def predict_ensemble(models, X, confidence_threshold=0.5):
    rf_model, xgb_model, lgbm_model, hist_gb_model = models
    
    rf_proba = rf_model.predict_proba(X)
    xgb_proba = xgb_model.predict_proba(X)
    lgbm_proba = lgbm_model.predict_proba(X)
    hist_gb_proba = hist_gb_model.predict_proba(X)

    mean_proba_segments_sq = (rf_proba**2 + xgb_proba**2 + lgbm_proba**2 + hist_gb_proba**2) / 4
    mean_proba_segments = (rf_proba + xgb_proba + lgbm_proba + hist_gb_proba) / 4

    confident_mask = np.max(mean_proba_segments_sq, axis=1) >= confidence_threshold
    
    if not np.any(confident_mask):
        avg_proba, pred_mean_label, avg_entropy = None, None, None
    else:
        avg_proba = np.mean(mean_proba_segments_sq[confident_mask], axis=0)
        pred_mean_encoded = rf_model.classes_[np.argmax(avg_proba)]
        pred_mean_label = le.inverse_transform([pred_mean_encoded])[0].split("\\")[-1].lower()
        avg_entropy = entropy(avg_proba, base=2)

    confident_mask_mv = np.max(mean_proba_segments, axis=1) >= confidence_threshold
    if not np.any(confident_mask_mv):
        pred_mv_label, vote_counts = None, {}
    else:
        X_conf = X[confident_mask_mv]
        rf_preds = le.inverse_transform(rf_model.predict(X_conf))
        xgb_preds = le.inverse_transform(xgb_model.predict(X_conf))
        lgbm_preds = le.inverse_transform(lgbm_model.predict(X_conf))
        hist_gb_preds = le.inverse_transform(hist_gb_model.predict(X_conf))
        
        combined_preds = np.concatenate([rf_preds, xgb_preds, lgbm_preds, hist_gb_preds])
        votes = Counter(combined_preds)
        vote_counts = {label.split("\\")[-1].lower(): count for label, count in votes.items()}
        pred_mv_label = votes.most_common(1)[0][0].split("\\")[-1].lower()

    return {'mean_proba': avg_proba, 'pred_mean_label': pred_mean_label, 'entropy': avg_entropy, 'pred_mv_label': pred_mv_label, 'vote_counts': vote_counts}

def composite_modification_score(original_df, modified_df, original_proba, modified_proba, target_class_index, weight_param, alpha=1.0, beta=1.0, gamma=1.0):
    if use_feature_engineering:
        original_df = engineer_features(original_df)
        modified_df = engineer_features(modified_df)
    n_orig, n_mod = len(original_df), len(modified_df)
    srr = (n_orig - n_mod) / n_orig if n_orig > 0 else 0
    
    orig_ids, mod_ids = set(original_df.index), set(modified_df.index)
    rem_length = original_df.loc[list(orig_ids - mod_ids), weight_param].sum()
    tot_length = original_df[weight_param].sum()
    wrs = rem_length / tot_length if tot_length > 0 else 0
    
    delta = modified_proba[target_class_index] - original_proba[target_class_index]
    delta_penalty = max(0, -delta)
    
    return alpha * srr + beta * wrs + gamma * delta_penalty

def correct_river_class_by_removing_segments(csv_path, target_class, models, features, confidence_threshold=0.5):
    df = pd.read_csv(csv_path, sep=';')
    if use_feature_engineering: df = engineer_features(df)
    original_df = df.copy()
    
    rf_model = models[0]
    classes_lower = [c.split("\\")[-1].lower() for c in le.inverse_transform(rf_model.classes_)]
    class_index = classes_lower.index(target_class.lower()) if target_class.lower() in classes_lower else None
    if class_index is None: return None, None

    while True:
        X_scaled = scaler.transform(imputer.transform(df[features].values))
        res = predict_ensemble(models, X_scaled, confidence_threshold)
        
        if res['pred_mean_label'] is None: break
        if res['pred_mean_label'] == target_class.lower(): return df, res['mean_proba']
        
        rf_p, xgb_p, lgbm_p, hgb_p = [m.predict_proba(X_scaled) for m in models]
        mean_p = (rf_p**2 + xgb_p**2 + lgbm_p**2 + hgb_p**2) / 4
        
        worst_idx = np.argmin(mean_p[:, class_index])
        df = df.drop(df.index[worst_idx]).reset_index(drop=True)
        if len(df) < 3: break
            
    return None, None


# 8. Evaluate all test rivers (with CMS)

In [9]:

test_items_dir = "data/test_items"
for f in os.listdir(test_items_dir):
    if f.endswith(".csv"):
        print(f"\n--- Prediction and CMS for {f} ---")
        path = os.path.join(test_items_dir, f)
        
        # Parse target class from filename (e.g. Amazon_dendritic.csv -> dendritic)
        target_class = f.replace(".csv", "").split("_")[-1].lower()
        
        df_river = pd.read_csv(path, sep=';')
        if use_feature_engineering: df_river = engineer_features(df_river)
        X_river = scaler.transform(imputer.transform(df_river[features].values))
        
        res = predict_ensemble(pseudo_models, X_river)
        print(f"Actual Class (from filename): {target_class}")
        print(f"Initial Prediction (Avg): {res['pred_mean_label']}")
        
        if res['pred_mean_label'] == target_class:
            print(">> Classification is already correct. CMS = 0.0000")
        else:
            print(">> Classification is wrong. Starting segment removal...")
            classes_list = [c.split("\\")[-1].lower() for c in le.inverse_transform(rf_base.classes_)]
            if target_class in classes_list:
                target_idx = classes_list.index(target_class)
                reduced_df, reduced_proba = correct_river_class_by_removing_segments(path, target_class, pseudo_models, features)
                if reduced_df is not None:
                    cms = composite_modification_score(df_river, reduced_df, res['mean_proba'], reduced_proba, target_idx, "NET_LENGTH", 0.5, 1.5, 1.0)
                    print(f">> Reduction successful! CMS = {cms:.4f}")
                else:
                    print(">> FAILURE: Unable to reduce to target class maintaining at least 3 segments.")
            else:
                print(f"Class {target_class} unknown.")



--- Prediction and CMS for Adige_angular.csv ---
Actual Class (from filename): angular
Initial Prediction (Avg): angular
>> Classification is already correct. CMS = 0.0000

--- Prediction and CMS for Amazon_dendritic.csv ---
Actual Class (from filename): dendritic
Initial Prediction (Avg): rectangular
>> Classification is wrong. Starting segment removal...
>> FAILURE: Unable to reduce to target class maintaining at least 3 segments.

--- Prediction and CMS for Brenta_trellis.csv ---
Actual Class (from filename): trellis
Initial Prediction (Avg): trellis
>> Classification is already correct. CMS = 0.0000

--- Prediction and CMS for Ebro_pinnate.csv ---
Actual Class (from filename): pinnate
Initial Prediction (Avg): pinnate
>> Classification is already correct. CMS = 0.0000

--- Prediction and CMS for Garonne_angular.csv ---
Actual Class (from filename): angular
Initial Prediction (Avg): angular
>> Classification is already correct. CMS = 0.0000

--- Prediction and CMS for Po_pinnate.cs